# 🎬 **ZiyanCorp UGC Video & Image AI Studio (Google Colab GPU Edition)**
### *Engine Generator Video 10s (Omni Flash / LTX-Video / Wan 2.1) & Foto Katalog Model*

---
### 🚀 **Panduan Penggunaan Cepat (1-Klik):**
1. Pastikan Runtime Colab menggunakan **GPU (T4 / A100)**: Klik menu `Runtime` > `Change runtime type` > pilih **`T4 GPU`** (Gratis).
2. Jalankan sel **Langkah 1 (Install Dependencies)**.
3. Jalankan sel **Langkah 2 (Load Model Video & Image)**.
4. Jalankan sel **Langkah 3 (Start Web App & API Bridge)**.
5. Klik link **`Gradio Public URL (https://xxxx.gradio.live)`** untuk langsung merender video dari HP atau laptop!

In [ ]:
# @title 🛠️ Langkah 1: Cek GPU & Install Library AI Video
!nvidia-smi

print("\n⏳ Menginstall Diffusers, PyTorch, Accelerate, Transformers, dan Gradio...")
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate sentencepiece protobuf
!pip install -q gradio fastapi uvicorn nest_asyncio pyngrok imageio imageio-ffmpeg

print("\n✅ Instalasi Selesai!")

In [ ]:
# @title 🧠 Langkah 2: Load AI Video Engine (LTX-Video / Diffusion)
import torch
from diffusers import LTXPipeline, AutoencoderKLLTXVideo
from diffusers.utils import export_to_video

print("⏳ Memuat model LTX-Video ke GPU...")
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    pipe = LTXPipeline.from_pretrained(
        "Lightricks/LTX-Video",
        torch_dtype=torch.bfloat16
    )
    pipe.to(device)
    pipe.enable_model_cpu_offload()
    print("\n✅ AI Video Engine Siap Digunakan!")
except Exception as e:
    print(f"[-] Error loading model: {e}")

In [ ]:
# @title 🎨 Langkah 3: Fungsi Render Video UGC 9:16 (10 Detik)
import time
import os

def generate_ugc_video(prompt, negative_prompt="low quality, blurry, distorted, deformed, cartoon", num_frames=121, num_steps=30, guidance_scale=3.0):
    print(f"\n🎬 Memulai render video: '{prompt[:50]}...'")
    t0 = time.time()
    
    # 9:16 Vertical Resolution (896x512)
    height = 896
    width = 512
    
    video_frames = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_frames=num_frames,
        num_inference_steps=num_steps,
        guidance_scale=guidance_scale,
    ).frames[0]
    
    output_path = "output_ugc_video.mp4"
    export_to_video(video_frames, output_path, fps=24)
    
    t1 = time.time()
    print(f"\n✅ Video Berhasil Dirender dalam {t1-t0:.1f} detik! File: {output_path}")
    return output_path

In [ ]:
# @title 🌐 Langkah 4: Buka Web Interface Publik (Bisa Diakses dari HP)
import gradio as gr

def ui_generate(prompt, neg_prompt, duration_sec):
    frames = int(duration_sec * 24)
    video_file = generate_ugc_video(prompt, neg_prompt, num_frames=min(frames, 121))
    return video_file

with gr.Blocks(theme=gr.themes.Soft(primary_hue="purple", neutral_hue="slate")) as demo:
    gr.Markdown("# 🎬 **ZiyanCorp AI Video Studio (Omni Flash / LTX GPU)**")
    gr.Markdown("Masukkan prompt dari aplikasi **NarasiKilat UGC Studio** di bawah ini untuk merender video MP4 10 detik secara gratis!")
    
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt Omni Flash / UGC Video (9:16 Vertical)",
                placeholder="Contoh: UGC TikTok video recorded on iPhone 15 Pro, vertical 9:16, photorealistic. A gorgeous Indonesian female influencer wearing Gamis Rayon walking in aesthetic cafe...",
                lines=4
            )
            neg_input = gr.Textbox(
                label="Negative Prompt",
                value="low quality, blurry, deformed hands, extra limbs, bad face, cartoon, 3d render, watermark",
                lines=2
            )
            duration_slider = gr.Slider(label="Durasi Video (Detik)", minimum=3, maximum=10, value=5, step=1)
            btn_submit = gr.Button("🚀 RENDER VIDEO MP4 SEKARANG", variant="primary")
            
        with gr.Column():
            video_output = gr.Video(label="Hasil Video MP4 10s")
            
    btn_submit.click(
        fn=ui_generate,
        inputs=[prompt_input, neg_input, duration_slider],
        outputs=[video_output]
    )

print("\n🚀 Membuka Web Interface... Link publik akan muncul di bawah ini:")
demo.launch(share=True, debug=True)